# 02 - Preprocessing

This notebook validates the preprocessing pipeline: missing value handling, categorical encoding, numeric scaling, train/test splitting, and SMOTE class balancing.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from imblearn.over_sampling import SMOTE

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from preprocess import (
    build_preprocessor,
    clean_telco_data,
    create_dataset_split,
    infer_feature_types,
    load_telco_data,
    save_split,
)

DATA_PATH = PROJECT_ROOT / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

In [ ]:
raw_df = load_telco_data(DATA_PATH)
clean_df = clean_telco_data(raw_df)
split = create_dataset_split(clean_df, test_size=0.2, random_state=42)

print(split.X_train.shape, split.X_test.shape)
split.y_train.value_counts(normalize=True).rename("train_target_rate")

## Feature Type Inference

In [ ]:
numeric_columns, categorical_columns = infer_feature_types(split.X_train)
print("Numeric columns:", numeric_columns)
print("Categorical columns:", categorical_columns)

## Imputation, Encoding, and Scaling

In [ ]:
preprocessor = build_preprocessor(split.X_train)
X_train_processed = preprocessor.fit_transform(split.X_train)
X_test_processed = preprocessor.transform(split.X_test)
feature_names = preprocessor.get_feature_names_out()

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)
print("First 15 encoded features:", feature_names[:15].tolist())
pd.DataFrame(X_train_processed, columns=feature_names).head()

In [ ]:
processed_frame = pd.DataFrame(X_train_processed, columns=feature_names)
processed_frame[numeric_columns].agg(["mean", "std"]).round(3)

## SMOTE Balancing

In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_processed, split.y_train)

print("Before SMOTE:")
display(split.y_train.value_counts().rename_axis("Churn").reset_index(name="customers"))
print("After SMOTE:")
display(pd.Series(y_resampled).value_counts().rename_axis("Churn").reset_index(name="customers"))
print("Resampled feature shape:", X_resampled.shape)

## Save Raw Train/Test Splits

In [ ]:
artifacts = save_split(split, PROJECT_ROOT / "data" / "processed")
artifacts